# TP Redes Bayesianas

### Nomes: Letícia Braga Esteves (202342209)
### Nomes: 
### Nomes: 
### Nomes: 
### Nomes: 



# Trabalho Prático III - Redes Bayesianas

## 1. Tarefa 1: Dados e pré-processamento

Acidentes de Trânsito da Polícia Rodoviária Federal (PRF), base de dados aprovada pelo professor João Pedro

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [64]:
# importando banco escolhido (Acidentes de Trânsito da Polícia Rodoviária Federal (PRF))
pd.set_option('display.max_columns', None)
df = pd.read_csv('data/datatran2026.csv', sep=";", decimal=",", encoding="latin1")


In [65]:
print(df.columns.tolist())

['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop']


In [66]:
# selecionando colunas para manter um numero de variaveis razoavel
colunas_selecionadas = [
    'dia_semana',
    'uf',
    'fase_dia',
    'condicao_metereologica',
    'tipo_pista',
    'causa_acidente',
    'tipo_acidente',
    'classificacao_acidente',
    'pessoas',
    'veiculos',
    'mortos'
]
df = df[colunas_selecionadas]

# removendo linhas com valores NaN (foi removido apenas 1 linha, agora temos 54894 linhas para 12 variaveis)
df = df.dropna()
print(len(df))
df.head()


54894


,dia_semana,uf,fase_dia,condicao_metereologica,tipo_pista,causa_acidente,tipo_acidente,classificacao_acidente,pessoas,veiculos,mortos
1,quinta-feira,MG,Pleno dia,Céu Claro,Simples,Condutor Dormindo,Colisão frontal,Com Vítimas Feridas,6,3,0
2,quinta-feira,SC,Amanhecer,Nublado,Dupla,Reação tardia ou ineficiente do condutor,Colisão lateral mesmo sentido,Com Vítimas Feridas,3,4,0
3,quinta-feira,DF,Pleno dia,Céu Claro,Dupla,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,3,3,0
4,quinta-feira,MT,Amanhecer,Sol,Simples,Transitar na contramão,Colisão frontal,Com Vítimas Feridas,3,5,0
5,quinta-feira,SP,Pleno dia,Céu Claro,Dupla,Velocidade Incompatível,Colisão com objeto,Com Vítimas Feridas,2,2,0


In [67]:
# discretizacao das variaveis contínuas em tres faixas por quantis (tercis)
df['pessoas_cat'] = pd.qcut(df['pessoas'], q=3, duplicates='drop')

df['veiculos_cat'] = pd.cut(
    df['veiculos'], 
    bins=[0, 1, 2, float('inf')], 
    labels=['1_veiculo', '2_veiculos', '3_ou_mais']
)

df['mortos_cat'] = pd.cut(
    df['mortos'], 
    bins=[-1, 0, 1, float('inf')], 
    labels=['Nenhum', '1_morto', '2_ou_mais']
)

df = df.drop(columns=['pessoas', 'veiculos', 'mortos'])


In [68]:
df.head()

,dia_semana,uf,fase_dia,condicao_metereologica,tipo_pista,causa_acidente,tipo_acidente,classificacao_acidente,pessoas_cat,veiculos_cat,mortos_cat
1,quinta-feira,MG,Pleno dia,Céu Claro,Simples,Condutor Dormindo,Colisão frontal,Com Vítimas Feridas,"(3.0, 88.0]",3_ou_mais,Nenhum
2,quinta-feira,SC,Amanhecer,Nublado,Dupla,Reação tardia ou ineficiente do condutor,Colisão lateral mesmo sentido,Com Vítimas Feridas,"(2.0, 3.0]",3_ou_mais,Nenhum
3,quinta-feira,DF,Pleno dia,Céu Claro,Dupla,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,"(2.0, 3.0]",3_ou_mais,Nenhum
4,quinta-feira,MT,Amanhecer,Sol,Simples,Transitar na contramão,Colisão frontal,Com Vítimas Feridas,"(2.0, 3.0]",3_ou_mais,Nenhum
5,quinta-feira,SP,Pleno dia,Céu Claro,Dupla,Velocidade Incompatível,Colisão com objeto,Com Vítimas Feridas,"(0.999, 2.0]",2_veiculos,Nenhum


In [69]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} categorias")

dia_semana: 7 categorias
uf: 27 categorias
fase_dia: 4 categorias
condicao_metereologica: 10 categorias
tipo_pista: 3 categorias
causa_acidente: 69 categorias
tipo_acidente: 17 categorias
classificacao_acidente: 3 categorias
pessoas_cat: 3 categorias
veiculos_cat: 3 categorias
mortos_cat: 3 categorias


In [78]:
top_causas = df['causa_acidente'].value_counts().nlargest(8).index
df['causa_acidente'] = df['causa_acidente'].where(
    df['causa_acidente'].isin(top_causas), 'Outras'
)

top_tipo = df['tipo_acidente'].value_counts().nlargest(8).index
df['tipo_acidente'] = df['tipo_acidente'].where(
    df['tipo_acidente'].isin(top_causas), 'Outras'
)

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54894 entries, 1 to 54894
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   dia_semana              54894 non-null  object  
 1   uf                      54894 non-null  object  
 2   fase_dia                54894 non-null  object  
 3   condicao_metereologica  54894 non-null  object  
 4   tipo_pista              54894 non-null  object  
 5   causa_acidente          54894 non-null  object  
 6   tipo_acidente           54894 non-null  object  
 7   classificacao_acidente  54894 non-null  object  
 8   pessoas_cat             54894 non-null  category
 9   veiculos_cat            54894 non-null  category
 10  mortos_cat              54894 non-null  category
dtypes: category(3), object(8)
memory usage: 3.9+ MB


### Perguntas:
#### 1. Dado que um acidente ocorreu durante a noite (Plena Noite), sob chuva (Chuva) e em pista simples, qual a probabilidade de haver vítimas fatais (mortos_cat != Nenhum)?
#### 2. Sabendo que ocorreu uma colisão frontal com vítimas fatais, quais são as causas de acidente mais prováveis ?
#### 3. Como o tipo de pista (simples vs. dupla) e o dia da semana influenciam a probabilidade de um acidente envolver 3 ou mais veículos?

## 2. Estrutura elicitada manualmente

Apresente as variáveis centrais, relações esperadas e justificativas.

## 3. Aprendizado estrutural

Compare Hill-Climbing com BIC e um segundo método.

## 4. Estimação de parâmetros e inferência

Estime CPTs e responda às perguntas formuladas com evidências variadas.

## 5. Explicabilidade

Escolha uma predição e explique-a por sensibilidade, caminhos de influência, Markov blanket ou contrafactuais.